In [219]:
!pip install transformers datasets evaluate rouge_score -q

In [220]:
!pip install -q datasets transformers sentencepiece

In [221]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

In [253]:
import pandas as pd

test_df = pd.read_parquet("test-sci.parquet")

print(test_df.head())
print(test_df.columns)
print("Total samples:", len(test_df))

                                            document  \
0  Prison Link Cymru had 1,099 referrals in 2015-...   
1  Officers searched properties in the Waterfront...   
2  Jordan Hill, Brittany Covington and Tesfaye Co...   
3  The 48-year-old former Arsenal goalkeeper play...   
4  Restoring the function of the organ - which he...   

                                             summary        id  
0  There is a "chronic" need for more housing for...  38264402  
1  A man has appeared in court after firearms, am...  34227252  
2  Four people accused of kidnapping and torturin...  38537698  
3  West Brom have appointed Nicky Hammond as tech...  36175342  
4  The pancreas can be triggered to regenerate it...  39070183  
Index(['document', 'summary', 'id'], dtype='object')
Total samples: 11334


In [254]:
df_100 = test_df.sample(n=10, random_state=42)

articles = df_100["document"].tolist()
reference_summaries = df_100["summary"].tolist()

print("Articles:", len(articles))
print("Summaries:", len(reference_summaries))

Articles: 10
Summaries: 10


In [224]:
!pip install transformers sentencepiece -q


In [225]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Using:", device)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Using: cuda


In [255]:
from tqdm import tqdm

generated_summaries = []

for article in tqdm(articles):

    inputs = tokenizer(
        article,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=130,
        min_length=30,
        num_beams=4,
        early_stopping=True
    )

    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    generated_summaries.append(generated_summary)

100%|██████████| 10/10 [00:08<00:00,  1.13it/s]


In [256]:
for i in range(3):
    print("\nREFERENCE SUMMARY:")
    print(reference_summaries[i])

    print("\nGENERATED SUMMARY:")
    print(generated_summaries[i])

    print("="*100)


REFERENCE SUMMARY:
The US government has imposed sanctions on 13 senior Venezuelan officials as pressure mounts on President Nicolás Maduro ahead of a controversial vote for a new constituent assembly.

GENERATED SUMMARY:
The sanctions freeze the US assets of those affected, and stop US entities from doing business with them. Those targeted include the interior minister and the head of the army. President Trump vowed "strong and swift economic actions" if Mr Maduro held the poll, due on Sunday.

REFERENCE SUMMARY:
Wales rugby great Gareth Edwards has been knighted by the Duke of Cambridge in recognition of a glittering sporting career and services to charity.

GENERATED SUMMARY:
The 68-year-old former scrum half won 53 caps for Wales from 1967 to 1978. Sir Gareth attended a ceremony at Windsor Castle on Thursday. He was named in the Queen's Birthday Honours list in June.

REFERENCE SUMMARY:
About 600 runners have taken part in the annual mountain race up Snowdon with a series of other

In [257]:
result_df = pd.DataFrame({
    "Article": articles,
    "Reference Summary": reference_summaries,
    "Generated Summary": generated_summaries
})

result_df.to_csv("bart_results.csv", index=False)

print("Saved successfully!")

Saved successfully!


In [229]:
from google.colab import files

files.download("bart_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [230]:
print(test_df.columns)

Index(['article', 'abstract'], dtype='object')


In [231]:
!pip install evaluate rouge_score -q

In [258]:
import evaluate

rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries
)

for key, value in results.items():
    print(f" '{key}': {value:.4f},")

 'rouge1': 0.1987,
 'rouge2': 0.0293,
 'rougeL': 0.1270,
 'rougeLsum': 0.1282,


In [233]:
# ==============================
# T5  SUMMARIZATION
# ==============================

!pip install transformers sentencepiece evaluate rouge_score -q

import torch
import pandas as pd
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [259]:
# Load T5

t5_model_name = "t5-base"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name).to(device)

t5_summaries = []

for article in tqdm(articles):

    article = "summarize: " + str(article)

    inputs = t5_tokenizer(
        article,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = t5_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=180,
        min_length=60,
        num_beams=8,
        length_penalty=1.2,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    summary = t5_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    t5_summaries.append(summary)


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:18<00:00,  1.88s/it]


In [260]:
for i in range(3):
    print("\nREFERENCE SUMMARY:")
    print(reference_summaries[i])

    print("\nGENERATED SUMMARY:")
    print(t5_summaries[i])

    print("="*100)


REFERENCE SUMMARY:
The US government has imposed sanctions on 13 senior Venezuelan officials as pressure mounts on President Nicolás Maduro ahead of a controversial vote for a new constituent assembly.

GENERATED SUMMARY:
sanctions freeze the assets of those affected, and stop US entities from doing business with them. those targeted include the interior minister and the head of the army. the vote is to choose the 545 members of a new constituent assembly. critics say the president is trying to cement a dictatorship.

REFERENCE SUMMARY:
Wales rugby great Gareth Edwards has been knighted by the Duke of Cambridge in recognition of a glittering sporting career and services to charity.

GENERATED SUMMARY:
the 68-year-old former scrum half won 53 caps for Wales from 1967 to 1978. he also won 10 caps for the British Lions' winning series in new Zealand and south africa. in 1974 he was named BBC Wales Sports Personality of the year.

REFERENCE SUMMARY:
About 600 runners have taken part in th

In [261]:
t5_df = pd.DataFrame({
    "Article": articles,
    "Reference Summary": reference_summaries,
    "Generated Summary": t5_summaries
})

t5_df.to_csv("t5_results.csv", index=False)

In [237]:
from google.colab import files

files.download("t5_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [262]:
import evaluate

rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=t5_summaries,
    references=reference_summaries  # ← FIX
)

for key, value in results.items():
    print(f" '{key}': {value:.4f},")

 'rouge1': 0.2241,
 'rouge2': 0.0423,
 'rougeL': 0.1362,
 'rougeLsum': 0.1374,


In [239]:
!pip install -q bert-score nltk

In [263]:
from transformers import pipeline
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

nli_model = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [264]:
test = nli_model(
    {"text": articles[0], "text_pair": generated_summaries[0]},
    truncation=True
)

print(test)

{'label': 'entailment', 'score': 0.7232100963592529}


In [265]:
print("Current dataset size:", len(articles))
print("First article:", articles[0][:200])
print("First BART summary:", generated_summaries[0])
print("First T5 summary:", t5_summaries[0])



Current dataset size: 10
First article: The sanctions freeze the US assets of those affected, and stop US entities from doing business with them.
Those targeted include the interior minister and the head of the army.
Last week, President Do
First BART summary: The sanctions freeze the US assets of those affected, and stop US entities from doing business with them. Those targeted include the interior minister and the head of the army. President Trump vowed "strong and swift economic actions" if Mr Maduro held the poll, due on Sunday.
First T5 summary: sanctions freeze the assets of those affected, and stop US entities from doing business with them. those targeted include the interior minister and the head of the army. the vote is to choose the 545 members of a new constituent assembly. critics say the president is trying to cement a dictatorship.


In [266]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [267]:

def is_summary_hallucinated(article, summary, chunk_size=400, overlap=50):
    sentences = sent_tokenize(str(summary))
    if len(sentences) == 0:
        return 0

    # Split article into overlapping word chunks
    words = article.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
        if i + chunk_size >= len(words):
            break

    for sent in sentences:
        # Check sentence against all chunks
        labels = []
        for chunk in chunks:
            result = nli_model(
                {"text": chunk, "text_pair": sent},
                truncation=True
            )
            labels.append(result["label"].lower())

        # If ANY chunk contradicts this sentence → hallucinated

        # If NO chunk entails it → also suspicious (hallucinated)
        if "entailment" not in labels:
            return 1

    return 0


bart_hallucinated_flags = []
t5_hallucinated_flags = []

for article, bart_sum, t5_sum in tqdm(
    zip(articles, generated_summaries, t5_summaries),
    total=len(articles)
):
    bart_hallucinated_flags.append(
        is_summary_hallucinated(article, bart_sum)
    )

    t5_hallucinated_flags.append(
        is_summary_hallucinated(article, t5_sum)
    )


bart_summary_hr = sum(bart_hallucinated_flags) / len(bart_hallucinated_flags)
t5_summary_hr = sum(t5_hallucinated_flags) / len(t5_hallucinated_flags)

print("BART hallucinated summaries:", sum(bart_hallucinated_flags))
print("BART summary-level hallucination rate:", bart_summary_hr)

print("T5 hallucinated summaries:", sum(t5_hallucinated_flags))
print("T5 summary-level hallucination rate:", t5_summary_hr)

100%|██████████| 10/10 [00:05<00:00,  1.88it/s]

BART hallucinated summaries: 5
BART summary-level hallucination rate: 0.5
T5 hallucinated summaries: 5
T5 summary-level hallucination rate: 0.5


In [268]:
# RESET and calculate again for current dataset
# RESET and calculate again for current dataset

bart_hallucination_scores = [
    is_summary_hallucinated(article, summary)
    for article, summary in tqdm(zip(articles, generated_summaries), total=len(articles))
]

t5_hallucination_scores = [
    is_summary_hallucinated(article, summary)
    for article, summary in tqdm(zip(articles, t5_summaries), total=len(articles))
]

bart_hallucination_rate = sum(bart_hallucination_scores) / len(bart_hallucination_scores)
t5_hallucination_rate = sum(t5_hallucination_scores) / len(t5_hallucination_scores)

print("BART Hallucination Rate:", bart_hallucination_rate)
print("T5 Hallucination Rate:", t5_hallucination_rate)

100%|██████████| 10/10 [00:02<00:00,  3.61it/s]

BART Hallucination Rate: 0.5
T5 Hallucination Rate: 0.5


In [269]:
for i, flag in enumerate(bart_hallucinated_flags):

    if flag == 1:

        print("="*120)
        print(f"ARTICLE {i+1}")

        print("\nREFERENCE SUMMARY:")
        print(reference_summaries[i])

        print("\nBART SUMMARY:")
        print(generated_summaries[i])

        print("\nHALLUCINATION DETECTED")
        print("="*120)

ARTICLE 3

REFERENCE SUMMARY:
About 600 runners have taken part in the annual mountain race up Snowdon with a series of other events being held to mark its 40th anniversary.

BART SUMMARY:
Emmanuel Manzi from Italy won the event at Llanberis, while Richard Roberts was the first Welsh competitor to finish. Organisers say the 10-mile race attracts some of the best mountain racers in Europe.

HALLUCINATION DETECTED
ARTICLE 4

REFERENCE SUMMARY:
Prominent Bahraini human rights activist Nabeel Rajab has been freed after serving two years in prison for his involvement in illegal protests.

BART SUMMARY:
Rajab, who heads the Bahrain Centre for Human Rights (BCHR), was convicted in 2012 of taking part in illegal gatherings. An appeals court later reduced his original three-year term by a year. He was one of several leading activists arrested by the authorities after pro-democracy protests erupted in 2011.

HALLUCINATION DETECTED
ARTICLE 6

REFERENCE SUMMARY:
Great Britain's Sir Bradley Wiggins

In [270]:
for i, flag in enumerate(t5_hallucinated_flags):

    if flag == 1:

        print("="*120)
        print(f"ARTICLE {i+1}")

        print("\nREFERENCE SUMMARY:")
        print(reference_summaries[i])

        print("\nT5 SUMMARY:")
        print(t5_summaries[i])

        print("\nHALLUCINATION DETECTED")
        print("="*120)

ARTICLE 1

REFERENCE SUMMARY:
The US government has imposed sanctions on 13 senior Venezuelan officials as pressure mounts on President Nicolás Maduro ahead of a controversial vote for a new constituent assembly.

T5 SUMMARY:
sanctions freeze the assets of those affected, and stop US entities from doing business with them. those targeted include the interior minister and the head of the army. the vote is to choose the 545 members of a new constituent assembly. critics say the president is trying to cement a dictatorship.

HALLUCINATION DETECTED
ARTICLE 2

REFERENCE SUMMARY:
Wales rugby great Gareth Edwards has been knighted by the Duke of Cambridge in recognition of a glittering sporting career and services to charity.

T5 SUMMARY:
the 68-year-old former scrum half won 53 caps for Wales from 1967 to 1978. he also won 10 caps for the British Lions' winning series in new Zealand and south africa. in 1974 he was named BBC Wales Sports Personality of the year.

HALLUCINATION DETECTED
ARTIC

In [271]:
from bert_score import score

P_bart, R_bart, F1_bart = score(
    generated_summaries,
    reference_summaries,
    lang="en",
    verbose=True
)

P_t5, R_t5, F1_t5 = score(
    t5_summaries,
    reference_summaries,
    lang="en",
    verbose=True
)

print("BART BERTScore F1:", F1_bart.mean().item())
print("T5 BERTScore F1:", F1_t5.mean().item())

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.22 seconds, 45.44 sentences/sec
BART BERTScore F1: 0.868566632270813
T5 BERTScore F1: 0.8656401634216309


In [272]:
rouge = evaluate.load("rouge")

bart_rouge = rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries
)

t5_rouge = rouge.compute(
    predictions=t5_summaries,
    references=reference_summaries
)

print("BART ROUGE:")
print(bart_rouge)

print("T5 ROUGE:")
print(t5_rouge)

BART ROUGE:
{'rouge1': np.float64(0.19873782312113303), 'rouge2': np.float64(0.02925852888788396), 'rougeL': np.float64(0.12702665016903508), 'rougeLsum': np.float64(0.12820746695975055)}
T5 ROUGE:
{'rouge1': np.float64(0.2240732154319111), 'rouge2': np.float64(0.042323047367210646), 'rougeL': np.float64(0.13616085192023253), 'rougeLsum': np.float64(0.13739004092620122)}


In [273]:
import pandas as pd

final_results = pd.DataFrame({
    "Model": ["BART", "T5"],

    "ROUGE-1": [
        bart_rouge["rouge1"],
        t5_rouge["rouge1"]
    ],

    "ROUGE-2": [
        bart_rouge["rouge2"],
        t5_rouge["rouge2"]
    ],

    "ROUGE-L": [
        bart_rouge["rougeL"],
        t5_rouge["rougeL"]
    ],

    "BERTScore F1": [
        F1_bart.mean().item(),
        F1_t5.mean().item()
    ],

    "Hallucination Rate": [
        sum(bart_hallucination_scores) / len(bart_hallucination_scores),
        sum(t5_hallucination_scores) / len(t5_hallucination_scores)
    ]
})

final_results

,Model,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore F1,Hallucination Rate
0,BART,0.198738,0.029259,0.127027,0.868567,0.5
1,T5,0.224073,0.042323,0.136161,0.865640,0.5


In [274]:
print(type(t5_summaries))
print(len(t5_summaries))

print(t5_summaries[:3])
print(generated_summaries[:3])

<class 'list'>
10
['sanctions freeze the assets of those affected, and stop US entities from doing business with them. those targeted include the interior minister and the head of the army. the vote is to choose the 545 members of a new constituent assembly. critics say the president is trying to cement a dictatorship.', "the 68-year-old former scrum half won 53 caps for Wales from 1967 to 1978. he also won 10 caps for the British Lions' winning series in new Zealand and south africa. in 1974 he was named BBC Wales Sports Personality of the year.", 'Emmanuel manzi from italy wins the event at Llanberis. he was the first Welsh competitor to finish. the 10-mile (16km) race attracts some of the best mountain racers in Europe. one couple got engaged after finishing the race.']
['The sanctions freeze the US assets of those affected, and stop US entities from doing business with them. Those targeted include the interior minister and the head of the army. President Trump vowed "strong and swi

In [252]:
print(len(articles))
print(len(generated_summaries))
print(len(t5_summaries))

10
10
10
